In [2]:
%pip install datasets matplotlib seaborn

   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 527.0/527.0 kB 8.0 MB/s  0:00:00
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   ---- ----------------------------------- 3.1/27.5 MB 14.8 MB/s eta 0:00:02
   -------- ------------------------------- 5.8/27.5 MB 13.8 MB/s eta 0:00:02
   ------------ --------------------------- 8.7/27.5 MB 13.9 MB/s eta 0:00:02
   ----------------- ---------------------- 12.1/27.5 MB 14.1 MB/s eta 0:00:02
   --------------------- ------------------ 14.9/27.5 MB 14.0 MB/s eta 0:00:01
   ------------------------- -------------- 17.6/27.5 MB 13.8 MB/s eta 0:00:01
   ---------------------------- ----------- 19.9/27.5 MB 13.5 MB/s eta 0:00:01
   -------------------------------- ------- 22.5/27.5 MB 13.3 MB/s eta 0:00:01
   ------------------------------------ --- 25.4/27.5 MB 13.3 MB/s eta 0:00:01
   ---------------------------------------- 27.5/27.5 MB 13.0 MB/s  0:00:0

## Importacion de dependencias y configuracion

In [3]:
import pandas as pd
import ollama
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# Configuracion visual para las graficas
sns.set_theme(style="whitegrid")

MMLU_CATEGORIAS = {
    "college_computer_science": 1,
    "computer_security": 2,
    "machine_learning": 3
}

## Funcion de traduccion robusta

In [4]:
def traducir_y_formatear(pregunta_eng, opciones_eng, respuesta_idx):
    letras = ["A", "B", "C", "D"]
    respuesta_letra = letras[respuesta_idx]
    
    prompt = f"""
    Eres un traductor tecnico experto en Ingenieria en Computacion.
    Traduce esta pregunta de opcion multiple del ingles al espanol tecnico de Mexico.
    
    Pregunta original: {pregunta_eng}
    Opciones:
    A) {opciones_eng[0]}
    B) {opciones_eng[1]}
    C) {opciones_eng[2]}
    D) {opciones_eng[3]}
    
    RESPONDE EXCLUSIVAMENTE CON UN JSON VALIDO. Sigue exactamente esta estructura:
    {{
        "contexto_practico": "La pregunta traducida de forma clara.",
        "opcion_a": "Traduccion de A",
        "opcion_b": "Traduccion de B",
        "opcion_c": "Traduccion de C",
        "opcion_d": "Traduccion de D",
        "respuesta_correcta": "{respuesta_letra}",
        "justificacion": "Genera una breve explicacion tecnica en espanol de por que la {respuesta_letra} es correcta."
    }}
    """
    
    try:
        response = ollama.chat(model="gemma2:9b", messages=[
            {'role': 'system', 'content': 'You output strictly valid JSON dictionaries.'},
            {'role': 'user', 'content': prompt},
        ])
        
        content = response['message']['content']
        
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0]
        elif "```" in content:
            content = content.split("```")[1].split("```")[0]
            
        return json.loads(content.strip())
    except Exception as e:
        print(f"[ERROR] Fallo en la traduccion o parseo JSON: {e}")
        return None

## Ingesta de datos

In [ ]:
nuevas_filas = []

print("Iniciando descarga y traduccion del dataset MMLU...")

for categoria, etiqueta in MMLU_CATEGORIAS.items():
    print(f"\nProcesando categoria: {categoria}")
    dataset = load_dataset("cais/mmlu", categoria, split="test")
    total_preguntas = len(dataset)
    
    for i, item in enumerate(dataset):
        if (i+1) % 10 == 0:
            print(f"Traduciendo reactivo [{i+1}/{total_preguntas}] de {categoria}...")
            # Guardado automatico silencioso
            pd.DataFrame(nuevas_filas).to_csv("data/dataset_mmlu_TEMP.csv", index=False)
            
        resultado = traducir_y_formatear(item['question'], item['choices'], item['answer'])
        
        if resultado:
            texto_combinado = f"{resultado.get('contexto_practico', '')}\nA) {resultado.get('opcion_a', '')}\nB) {resultado.get('opcion_b', '')}\nC) {resultado.get('opcion_c', '')}\nD) {resultado.get('opcion_d', '')}"
            respuesta_combinada = f"Respuesta: {resultado.get('respuesta_correcta', '')} - {resultado.get('justificacion', '')}"
            
            nuevas_filas.append({
                "texto": texto_combinado,
                "respuesta": respuesta_combinada,
                "etiqueta": etiqueta,
                "area_origen": f"MMLU_{categoria}"
            })

df_mmlu = pd.DataFrame(nuevas_filas)
print(f"\nProceso finalizado. Total de reactivos traducidos: {len(df_mmlu)}")

Iniciando descarga y traduccion del dataset MMLU...

Procesando categoria: college_computer_science


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

college_computer_science/test-00000-of-0(…):   0%|          | 0.00/28.1k [00:00<?, ?B/s]

college_computer_science/validation-0000(…):   0%|          | 0.00/6.25k [00:00<?, ?B/s]

college_computer_science/dev-00000-of-00(…):   0%|          | 0.00/6.81k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

## Analisis Visual

In [ ]:
# Grafica 1: Distribucion de categorias importadas
plt.figure(figsize=(10, 6))
ax = sns.countplot(data=df_mmlu, x='area_origen', palette='viridis')
plt.title("Distribucion de Preguntas por Categoria (MMLU)", fontsize=14, pad=15)
plt.xlabel("Categoria de Origen", fontsize=12)
plt.ylabel("Cantidad de Preguntas Generadas", fontsize=12)
plt.xticks(rotation=15)

# Agregar las etiquetas de datos sobre las barras
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 8), textcoords='offset points')

plt.tight_layout()
plt.show()

# Vista previa de la calidad de los datos
print("\nVista previa de los primeros 3 registros:")
display(df_mmlu.head(3))

In [4]:
import pandas as pd

# 1. Cargar los datos (nota el ../ al inicio de las rutas)
df_original = pd.read_csv("../data/dataset_ceneval_final.csv")
df_mmlu = pd.read_csv("../data/dataset_mmlu_traducido_final.csv")

print("Archivos cargados correctamente.")

# 2. Unir (Concatenar) los datasets
df_super = pd.concat([df_original, df_mmlu], ignore_index=True)

# 3. Limpiar nulos y mezclar
df_super = df_super.dropna(subset=['respuesta'])
df_super = df_super.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Guardar el archivo maestro (también con ../)
ruta_final = "../data/dataset_ceneval_super_final.csv"
df_super.to_csv(ruta_final, index=False, encoding='utf-8')

print(f"Fusión completada. Tamaño final: {len(df_super)} preguntas.")

Archivos cargados correctamente.
Fusión completada. Tamaño final: 2020 preguntas.
